In [3]:
import json
from pprint import pprint
from src.ACMPipelineClass import *
from src.brand_configs import CommonPipelineConfig, create_pipeline_for_brand

# AMC Custom Model Training & Inference Pipeline

This notebook provides an automated pipeline for training and running inference with custom machine learning models on Amazon Marketing Cloud (AMC) data. The system supports two distinct use cases: **Audience Creation** and **Measurement (MMM)**.

## Use Case Types
- **Audience** (`use_case_type = 'audience'`): Creates datasets for audience targeting, outputs go to AMC for audience creation
- **Measurement** (`use_case_type = 'measurement'`): Supports measurement (ex: MMM), runs MLIC, trains the model, and saves output directly to S3 bucket for analysis

## Recommended Workflow
It is recommended that for your first run - you do an end-to-end "dry run" deployment. The only configs that should be changed are the instanceId, membershipId and AccountId in the class. All others can stay the same. Once comfortable with the dry run process, begin modifications to customize model (details in Part 3 of readme.md)

## What This Pipeline Does

**Training Phase:**
1. **Containerization**: Builds your custom ML code into Docker containers and pushes them to AWS ECR (Elastic Container Registry)
2. **Data Preparation**: Creates ML Input Channels by executing your SQL queries against AMC data
3. **Model Training**: Runs your training algorithm on AWS SageMaker using the prepared data
4. **Model Storage**: Saves the trainedModelId for use in Inference. NOTE - you will not have access to this model
   - **Measurement use cases**: Saves results directly to S3 bucket for measurement analysis

**Inference Phase:**
1. **Data Processing**: Prepares new data using your inference SQL queries
2. **Prediction Generation**: Runs your trained model against new data to generate predictions
3. **Output Delivery**: 
   - **Audience use cases**: Creates new dataset in AMC with your model predictions for audience targeting  

## Key Customization Points
- **Use Case Type**: Choose between 'audience' or 'measurement' workflows
- **SQL Queries**: Define what data to extract from AMC for training and inference
- **ML Code**: Customize `ml/code/train/train.py` and `ml/code/infer/infer.py` with your machine learning algorithms
- **Compute Resources**: Configure instance types and sizes based on your data volume
- **Dependencies**: Add required Python packages in `requirements.txt` files

# Part 0: List Existing Algorithms

Before creating new algorithms, you can browse all existing algorithm configurations without any setup or initialization. This helps you:

- **List existing algorithms** that you can reuse
- **Avoid naming conflicts** with existing algorithms

In [4]:
import os
os.environ["DYNAMODB_TABLE_CONFIGS"] = "amc-acm-custom-model-stack-ca-ACM-Algorithm-Configs"
os.environ["DYNAMODB_TABLE_EXECUTIONS"] = "amc-acm-custom-model-stack-ca-PipeLine-Executions"

In [5]:
# One-time kernel fix after installing botocore[crt] while notebook is already running
import importlib
import awscrt
import boto3
import botocore.credentials as _bc_credentials

importlib.reload(_bc_credentials)
boto3.setup_default_session()

print("awscrt version:", awscrt.__version__)
print("botocore CRT dependency loaded:", getattr(_bc_credentials, "EC", None) is not None)

awscrt version: 0.36.0
botocore CRT dependency loaded: True


In [24]:
all_configs = ACMPipeline.list_algorithms()

# Filter by algorithm name
# ACMPipeline.list_algorithms(algorithm_name='dry-run-audience-amcinstanceid')

# Get specific configuration
# ACMPipeline.list_algorithms(algorithm_name='my-algorithm', instance_id='amc123abc')

Found 4 algorithm configuration(s):

--- Configuration 1 ---
Algorithm Name: ntb_audience_ca_liv_v1
Instance ID: amcy3dkn05n
AMC Account ID: ENTITY3F46DUUW73OQH
Membership Identifier: 90d28e98-e6e3-40ef-a311-07456c4c8d40
Region: us-east-1
Use Case Type: audience
Base Path: /home/ec2-user/SageMaker/ACM_Model_Creation/ml/ntb_code
Metrics Used: ntb_metrics_definition
CMAA Config Used: dry_run_config
Schema Name: user_level_ntb_campaign_schema
Last Updated: 2026-10-12T17:27:43.456919

--- Configuration 2 ---
Algorithm Name: ntb_audience_ca_dove_v2
Instance ID: amcy3dkn05n
AMC Account ID: ENTITY3F46DUUW73OQH
Membership Identifier: 90d28e98-e6e3-40ef-a311-07456c4c8d40
Region: us-east-1
Use Case Type: audience
Base Path: /home/ec2-user/SageMaker/ACM_Model_Creation/ml/ntb_code
Metrics Used: ntb_metrics_definition
CMAA Config Used: dry_run_config
Schema Name: user_level_ntb_campaign_schema
Last Updated: 2026-10-15T13:31:35.365254

--- Configuration 3 ---
Algorithm Name: ntb_audience_ca_dove_v1


# Part 1: Configure Your Algorithm

## Step 1.1 - Algorithm Name
Choose a unique identifier for your machine learning algorithm. This name acts as a primary key and will be used to:
- Create AWS resources (ECR repositories, SageMaker models, etc.)
- Track executions in DynamoDB
- Organize your model artifacts

## Step 1.2 - AMC Details
**Required Updates**
- instance_id = Your AMC instanceId: amcxyz123 - this can be found via the AMC Console.
- amc_account_id = Your AMC accountId: ENTITYABCDE12345 - This can be found via the AMC Console. This is your AMC Account Entity ID, not an advertiser ID. This should match the URL when in the target AMC instance. EX:  https://advertising.amazon.com/marketing-cloud?entityId=ENTITY*ABC12345*. 

- membershipIdentifier = AWS Clean Rooms Collaboration Memebership ID - This can be found going to AWS -> AWS Clean Rooms -> Your instance collaboration (will have your instance id in the name) -> Details -> Membership ID

**Optional Updates**
- Region = Region currently using in AWS
- Verbose = Print additional details on all processes

### Configuration Parameters:

- **use_case_type** = 'audience' or 'measurement'
  - **Purpose**: Determines the workflow type and output destination
  - **Values**: 
    - `'audience'`: Creates datasets for audience targeting, outputs go to AMC
    - `'measurement'`: Supports measurement, outputs saved to S3 bucket
  - **Use case**: Choose based on whether you're building audiences or running measurement analysis

- **base_path** = '/home/ec2-user/SageMaker/ACM_Model_Creation/ml/audience_dry_run_code' or '/home/ec2-user/SageMaker/ACM_Model_Creation/ml/mmm_measurement_dry_run_code'
  - **Purpose**: Points to where your Dockerfile and train/infer .py files are located
  - **How to update**: 
    - For audience use cases: `/ml/audience_dry_run_code`
    - For measurement use cases: `/ml/mmm_measurement_dry_run_code`
  - **Use case**: Organize models by use case type with separate code directories

- **metrics_name** = 'dry_run_definitions' or 'mmm_dry_run_definition'
  - **Purpose**: References the JSON file in `API/metrics_definitions/` that defines your model metrics
  - **How to update**: 
    - For audience use cases: `'dry_run_definitions'`
    - For measurement use cases: `'mmm_dry_run_definition'`
  - **Note**: These are the fields that will be printed to CloudWatch, your train.py will need to align with these values
  - **Use case**: Different use case types require different metric definitions

- **cmaa_config_name** = 'dry_run_config'
  - **Purpose**: References the JSON file in `API/cmaa/` that sets max sizes for trainedModelExports and trainedModelInferenceJobs
  - **How to update**: Create a new .json file in the cmaa folder and reference its name (without .json extension)
  - **Use case**: Different models may have different resource requirements

- **schema_name** = 'dry_run_amc_schema'
  - **Purpose**: References the JSON file in `API/schema/` that defines your AMC data schema. You will need to align your inference.py to align with this schema. 
  - **How to update**: Create a new .json file in the schema folder and reference its name (without .json extension)
  - **Use case**: Different models may work with different output formats 


**Important**: Use the same name to update an existing algorithm, or choose a new name to create separate resources.

## Environment Variables
The CloudFormation template automatically sets up these AWS service connections for you.

In [6]:
from pathlib import Path

# Select which brand config to run (examples: "dmc", "liv")
brand_key = "dmc"

# Pull in service names created by the CloudFormation Template
cloud_formation_env_vars = get_cloudformation_env_vars()

# Workspace-local config paths for this notebook environment
project_root = Path.cwd()
schema_file_path = str(project_root / "API" / "schema")
metrics_definition_directory = str(project_root / "API" / "metrics_definitions")
cmaa_config_directory = str(project_root / "API" / "cmaa")
local_base_path = str(project_root / "ml" / "ntb_code")

# Shared (non-brand-specific) settings
common_config = CommonPipelineConfig(
    algorithm_prefix="ntb_audience",
    instance_id="amcy3dkn05n",
    membership_identifier="90d28e98-e6e3-40ef-a311-07456c4c8d40",
    amc_account_id="ENTITY3F46DUUW73OQH",
    use_case_type="audience",
    base_path=local_base_path,
    metrics_name="ntb_metrics_definition",
    cmaa_config_name="dry_run_config",
    schema_name="user_level_ntb_campaign_schema",
)

pipeline_bundle = create_pipeline_for_brand(
    brand_key=brand_key,
    common=common_config,
    cloud_formation_env_vars=cloud_formation_env_vars,
)

pipeline = pipeline_bundle["pipeline"]
brand_config = pipeline_bundle["brand_config"]
acm_algorithm_name = pipeline_bundle["algorithm_name"]

print("Selected brand:", brand_key)
print("Configured algorithm:", acm_algorithm_name)
print("Using base_path:", local_base_path)

Using base_path: /Users/ecrawfor/aws-custom-models-ca/ml/ntb_code


## Step 1.3: Build Brand Campaign Allowlist (Auto-Injection into SQL)

Use this step to pull campaigns from your Glue brand mapping table for the brands you want to include.

How to use:
1. Set `brand_key` in Step 1.2.
2. Run the next cell to build `campaign_filter_clause`.
3. Continue to training/inference payload cells; they now inject this filter into SQL automatically.

This avoids manual monthly SQL edits and prevents unmapped campaigns from entering model runs.

In [7]:
# Part 1.3: Build campaign allowlist from Glue/Athena
# for selected brands and target market

import boto3
import time
import re

if "brand_config" not in globals():
    raise ValueError("Run Step 1.2 first to load brand configuration.")

# Loaded from API/brands/<brand_key>.json
target_brands = list(brand_config.target_brands)

# Filter the Athena mapping table by market
target_market = str(brand_config.target_market)

# Optional keyword fallback list
# Case-insensitive LIKE against campaign text.
fallback_brand_keywords = list(brand_config.fallback_brand_keywords)

# Glue/Athena mapping table settings
brand_mapping_database = "custom-audience-pilot-db"
brand_mapping_table = "master_brand_mapping"
brand_mapping_workgroup = "primary"
brand_mapping_region = "us-east-1"

# Athena column names
brand_mapping_campaign_column = "campaign_name"
brand_mapping_brand_column = "brand"
brand_mapping_market_column = "market"

# Preferred Athena output bucket for query results
athena_output_bucket = "master-data-custom-audiences"

# Map known AMC instance IDs to market folder names.
# Add more mappings over time; fallback is the instance ID itself.
instance_market_map = {
    "amcy3dkn05n": "CA",
}


def _slug(value):
    text = re.sub(r"[^a-zA-Z0-9]+", "-", str(value).strip().lower())
    text = text.strip("-")
    return text or "unknown"


def _sql_quote(value):
    return "'" + str(value).replace("'", "''") + "'"


def _quote_identifier(identifier):
    # Athena/Trino identifier quoting using double quotes
    return '"' + str(identifier).replace('"', '""') + '"'


def _resolve_effective_athena_output(
    preferred_output_location,
    workgroup,
    region,
):
    athena = boto3.client("athena", region_name=region)

    try:
        wg = athena.get_work_group(
            WorkGroup=workgroup
        )["WorkGroup"]

        cfg = wg.get("Configuration", {})

        enforce = bool(
            cfg.get("EnforceWorkGroupConfiguration", False)
        )

        wg_output = cfg.get(
            "ResultConfiguration", {}
        ).get("OutputLocation")

    except Exception as exc:
        if preferred_output_location:
            return (
                preferred_output_location,
                "workgroup_lookup_denied_using_client_output",
            )

        raise ValueError(
            "Cannot inspect Athena workgroup and no preferred "
            "output location is set. Set athena_output_bucket "
            "or grant athena:GetWorkGroup."
        ) from exc

    # If enforced, Athena ignores client-side output location values.
    if enforce and wg_output:
        return wg_output, "workgroup_enforced"

    if preferred_output_location:
        return (
            preferred_output_location,
            "not_enforced_client_output",
        )

    if wg_output:
        return (
            wg_output,
            "not_enforced_workgroup_default",
        )

    raise ValueError(
        "No Athena output location found. Set athena_output_bucket "
        "or configure a workgroup output location."
    )


def _run_athena_query(
    query,
    database,
    workgroup,
    region,
    output_location=None,
    poll_seconds=3,
):
    athena = boto3.client("athena", region_name=region)

    start_args = {
        "QueryString": query,
        "QueryExecutionContext": {
            "Database": database
        },
        "WorkGroup": workgroup,
    }

    if output_location:
        start_args["ResultConfiguration"] = {
            "OutputLocation": output_location
        }

    qid = athena.start_query_execution(
        **start_args
    )["QueryExecutionId"]

    while True:
        execution = athena.get_query_execution(
            QueryExecutionId=qid
        )["QueryExecution"]

        state = execution["Status"]["State"]

        if state in {"SUCCEEDED", "FAILED", "CANCELLED"}:
            break

        time.sleep(poll_seconds)

    if state != "SUCCEEDED":
        reason = execution["Status"].get(
            "StateChangeReason",
            "Unknown Athena error",
        )

        raise RuntimeError(
            f"Athena query failed ({state}): {reason}"
        )

    campaigns = []
    next_token = None

    while True:
        kwargs = {
            "QueryExecutionId": qid
        }

        if next_token:
            kwargs["NextToken"] = next_token

        page = athena.get_query_results(**kwargs)

        rows = page.get(
            "ResultSet", {}
        ).get("Rows", [])

        # Skip the Athena result header on the first page
        if next_token is None and rows:
            rows = rows[1:]

        for row in rows:
            data = row.get("Data", [])

            if data:
                value = data[0].get("VarCharValue")

                if value:
                    campaigns.append(value)

        next_token = page.get("NextToken")

        if not next_token:
            break

    return sorted(set(campaigns)), qid


if not target_brands:
    raise ValueError(
        "target_brands is empty. Add at least one brand "
        "before continuing."
    )

if not str(target_market).strip():
    raise ValueError(
        "target_market is empty. Set a market such as 'UK'."
    )

if "pipeline" not in globals():
    raise ValueError(
        "Run Step 1.2 first so pipeline/instance context "
        "is available."
    )

current_instance_id = getattr(
    pipeline,
    "instance_id",
    None,
)

if not current_instance_id:
    raise ValueError(
        "Could not detect pipeline.instance_id. "
        "Confirm Step 1.2 completed successfully."
    )

market_code = instance_market_map.get(
    current_instance_id,
    current_instance_id,
)

brand_folder = "_".join(
    sorted(
        {
            _slug(brand)
            for brand in target_brands
            if str(brand).strip()
        }
    )
) or "all-brands"

preferred_output_prefix = (
    f"athena-results/"
    f"{_slug(target_market)}/"
    f"{brand_folder}/"
)

preferred_output_location = (
    f"s3://{athena_output_bucket}/"
    f"{preferred_output_prefix}"
)

effective_output_location, output_mode = (
    _resolve_effective_athena_output(
        preferred_output_location=preferred_output_location,
        workgroup=brand_mapping_workgroup,
        region=brand_mapping_region,
    )
)

# Only pass output location explicitly when the workgroup
# is not enforcing its own value.
query_output_override = (
    effective_output_location
    if output_mode != "workgroup_enforced"
    else None
)

campaign_col_quoted = _quote_identifier(
    brand_mapping_campaign_column
)

brand_col_quoted = _quote_identifier(
    brand_mapping_brand_column
)

market_col_quoted = _quote_identifier(
    brand_mapping_market_column
)

brand_values_sql = ", ".join(
    _sql_quote(str(brand).strip().lower())
    for brand in target_brands
    if str(brand).strip()
)

target_market_sql = _sql_quote(
    str(target_market).strip().lower()
)

campaign_query = f"""
SELECT DISTINCT
    {campaign_col_quoted} AS campaign
FROM "{brand_mapping_database}"."{brand_mapping_table}"
WHERE LOWER(TRIM(CAST({brand_col_quoted} AS VARCHAR)))
      IN ({brand_values_sql})
  AND LOWER(TRIM(CAST({market_col_quoted} AS VARCHAR)))
      = {target_market_sql}
  AND {campaign_col_quoted} IS NOT NULL
  AND TRIM(CAST({campaign_col_quoted} AS VARCHAR)) <> ''
"""

campaign_allowlist, athena_query_id = _run_athena_query(
    query=campaign_query,
    database=brand_mapping_database,
    workgroup=brand_mapping_workgroup,
    region=brand_mapping_region,
    output_location=query_output_override,
    poll_seconds=3,
)

campaign_allowlist_sql = (
    ", ".join(
        _sql_quote(campaign)
        for campaign in campaign_allowlist
    )
    if campaign_allowlist
    else ""
)


def build_campaign_filter_expression(campaign_col_expr):
    like_clauses = []

    for keyword in fallback_brand_keywords:
        keyword_clean = str(keyword).strip().lower()

        if not keyword_clean:
            continue

        keyword_sql = keyword_clean.replace("'", "''")

        like_clauses.append(
            f"LOWER(CAST({campaign_col_expr} AS VARCHAR)) "
            f"LIKE '%{keyword_sql}%'"
        )

    mapped_clause = (
        f"{campaign_col_expr} IN ({campaign_allowlist_sql})"
        if campaign_allowlist_sql
        else "FALSE"
    )

    like_clause = (
        " OR ".join(like_clauses)
        if like_clauses
        else "FALSE"
    )

    return f"(({mapped_clause}) OR ({like_clause}))"


campaign_filter_condition_di = (
    build_campaign_filter_expression("di.campaign")
)

campaign_filter_condition_campaign = (
    build_campaign_filter_expression("campaign")
)

print("Athena query used:")
print(campaign_query)
print(f"Current instance_id: {current_instance_id}")
print(f"Instance market mapping: {market_code}")
print(f"Target market filter: {target_market}")
print(f"Brand folder: {brand_folder}")
print(f"Mapping table: {brand_mapping_table}")
print(f"Preferred output location: {preferred_output_location}")
print(f"Effective output mode: {output_mode}")
print(f"Effective output location: {effective_output_location}")
print(f"Athena QueryExecutionId: {athena_query_id}")
print(f"Brands: {target_brands}")
print(
    "Campaigns found for selected brands and market: "
    f"{len(campaign_allowlist)}"
)

if len(campaign_allowlist) == 0:
    print(
        "No mapped campaigns found for the selected brands and "
        "market. SQL will rely on the case-insensitive LIKE "
        "fallback only."
    )

print(
    "Ready: campaign_filter_condition_di and "
    "campaign_filter_condition_campaign"
)

Athena query used:

SELECT DISTINCT
    "campaign_name" AS campaign
FROM "custom-audience-pilot-db"."master_brand_mapping"
WHERE LOWER(TRIM(CAST("brand" AS VARCHAR)))
      IN ('dove')
  AND LOWER(TRIM(CAST("market" AS VARCHAR)))
      = 'ca'
  AND "campaign_name" IS NOT NULL
  AND TRIM(CAST("campaign_name" AS VARCHAR)) <> ''

Current instance_id: amcy3dkn05n
Instance market mapping: CA
Target market filter: CA
Brand folder: dove
Mapping table: master_brand_mapping
Preferred output location: s3://master-data-custom-audiences/athena-results/ca/dove/
Effective output mode: not_enforced_client_output
Effective output location: s3://master-data-custom-audiences/athena-results/ca/dove/
Athena QueryExecutionId: 4372a84e-6a0b-47cd-9940-014fbeaebccc
Brands: ['Dove']
Campaigns found for selected brands and market: 1049
Ready: campaign_filter_condition_di and campaign_filter_condition_campaign


# Part 1.4: Using Existing Algorithms

This section shows how to discover and reuse existing algorithm configurations from DynamoDB. This is useful when you want to:
- **Quick setup** without manually entering all parameters

In [27]:
##### Load existing configuration by algorithm name and instance ID
##### Replace these values with the algorithm you want to load:
cloud_formation_env_vars = get_cloudformation_env_vars()


pipeline = ACMPipeline.from_existing_config(
     acm_algorithm_name='ntb_audience_ca_dove_v2',  # Replace with your algorithm name
     instance_id='amcy3dkn05n',                              # Replace with your instance ID
     **cloud_formation_env_vars
 )

acm_algorithm_name = pipeline.acm_algorithm_name

Successfully loaded: ntb_audience_ca_dove_v2

--- Loaded Configuration ---
Algorithm Name: ntb_audience_ca_dove_v2
Instance ID: amcy3dkn05n
AMC Account ID: ENTITY3F46DUUW73OQH
Membership Identifier: 90d28e98-e6e3-40ef-a311-07456c4c8d40
Region: us-east-1
Use Case Type: audience
Base Path: /home/ec2-user/SageMaker/ACM_Model_Creation/ml/ntb_code
Metrics Name: ntb_metrics_definition
CMAA Config Name: dry_run_config
Schema Name: user_level_ntb_campaign_schema
Last Updated: 2026-10-15T13:31:35.365254


# Part 2: Model Training

## Step 2.1: Configure Training Parameters

Define how your model should be trained by configuring:
- **Compute Resources**: Instance type and size for training
- **Data Sources**: SQL queries to extract training data from AMC
- **Time Windows**: Date ranges for your training data
- **vol_size**: The size of the AWS EBS (Elastic Block Store) volume in GB. This volume stores both the input data from ML Input Channels and any model artifacts written during training. You should size this based on the approximateSizeOnDisk of your ML Input Channels plus additional storage needed for model artifacts.
- **timeout**: The maximum runtime in seconds for the training job (specified in stoppingCondition.maxRuntimeInSeconds). The training job will be terminated if it exceeds this duration.

Optional fields:
- **user_level_targeting**: When set to true, AMC will validate that the training datasets being used support audience creation. Set this to true if you plan to use the model outputs for creating audiences.
- **environment**: Key-value pairs that will be made available as environment variables to your training container. In the example, 'SM_CHANNEL_TRAIN': 'mlic_name' maps the ML Input Channel name to a specific training channel.
- **hyper_parameters**: Key-value pairs that specify model hyperparameters. These will be made available to your training container to control model behavior. In the example, "learning_rate": "0.001" sets the learning rate hyperparameter to 0.001.

#### Usage
- ML Input Channel are created by providing an AMC SQL query and associated time window to the AMC API. ML Input Channel stores the query results (data), enabling the data to later be used for training and inferencing. 
- The AMC SQL Query to create an ML Input Channel does not require aggregation (GROUP BY) in the final SELECT statement.
- A single training job can accept up to 20 ML Input Channels. Queries over long time periods can be split into multiple ML Input Channels. 
- ML Input Channel resources have a TTL of 28 days. After 28 days have passed, the ML Input Channel will become INACTIVE, which is reflected by the status field in the API response. An INACTIVE ML Input Channel cannot be leveraged for training or inferencing. 
- Each ML Input Channel can be associated with a single Configured Model Algorithm Association (CMAA).

#### Tips 
- **For Audience Use Cases**: Trained Models and Inference Jobs intended for audience creation must use **_for_audiences variants of AMC datasources only. 
- **For Measurement Use Cases**: Can use regular AMC datasources (no _for_audiences suffix required)
- By specifying enforceUserLevelTargeting=True in the API call, AMC will check if ML Input Channel created is allowed for audience creation.
- Trained Models can only be created with ML Input Channel(s) which are in ACTIVE state. 
- Trained Models have a TTL of 30 days, AMC will automatically delete Trained Model resources once the TTL has expired. Model evaluation can be performed during the training job execution. A typical approach is to (1) separate a portion of the training data for testing, (2) leverage the trained model to make predictions on the test data, (3) Log custom metrics (e.g., RMSE, AUC) in the container. If you setup a service role for Clean Rooms to push custom metrics to Cloudwatch, you can access custom metrics in Cloudwatch once the training job is complete.

In [8]:
# Payload for the training pipeline
if "campaign_filter_condition_di" not in globals():
    raise ValueError(
        "Run Part 1.3 first to build SQL campaign filter conditions."
    )

# Fallback: if a separate click-condition variable is not defined,
# reuse the DI campaign condition.
if "campaign_filter_condition_campaign" not in globals():
    campaign_filter_condition_campaign = campaign_filter_condition_di

# Label look-forward window sensitivity test:
if "follow_up_days" not in globals():
  follow_up_days = 28

if "eligible_user_sample_rate" not in globals():
    eligible_user_sample_rate = 0.30

if "max_eligible_users" not in globals():
    max_eligible_users = 2500000

if "negative_keep_rate_sql" not in globals():
    negative_keep_rate_sql = 1.0

# EU in-market segment profile control.
if "segment_marketplace_id" not in globals():
  segment_marketplace_id = 7

train_sql_base = f'''
WITH
  cutoff AS (
    SELECT EPOCH_TO_UTC_TIMESTAMP(
      SECONDS_BETWEEN(CAST('1970-01-01' AS DATE), BUILT_IN_PARAMETER('TIME_WINDOW_END'))
      - (86400 * {follow_up_days})
    ) AS cutoff_date
  ),

  filtered_brand_impressions AS (
    SELECT
      di.user_id,
      di.impression_dt_utc,
      di.impressions
    FROM
      TABLE(EXTEND_TIME_WINDOW('dsp_impressions_for_audiences', 'P30D', 'P0D')) di
    WHERE
      di.user_id IS NOT NULL
      AND {campaign_filter_condition_di}
  ),

  filtered_brand_clicks AS (
    SELECT
      dc.user_id,
      dc.click_dt_utc,
      dc.clicks
    FROM
      TABLE(
        EXTEND_TIME_WINDOW(
          'dsp_clicks_for_audiences',
          'P30D',
          'P0D'
        )
      ) dc
    WHERE
      dc.user_id IS NOT NULL
      AND {campaign_filter_condition_campaign}
  ),

  eligible_users_base AS (
    SELECT DISTINCT
      fbi.user_id
    FROM filtered_brand_impressions fbi
    CROSS JOIN cutoff c
    WHERE SECONDS_BETWEEN(fbi.impression_dt_utc, c.cutoff_date) BETWEEN 0 AND 86400 * 30
  ),

  eligible_users AS (
    SELECT user_id
    FROM eligible_users_base
    WHERE random() < {eligible_user_sample_rate}
    LIMIT {max_eligible_users}
  ),

  ntb_users AS (
    SELECT
      user_id,
      MIN(
        CASE
          WHEN new_to_brand = TRUE THEN event_dt_utc
        END
      ) AS first_ntb_purchase_date,
      MAX(
        CASE
          WHEN new_to_brand = TRUE THEN 1
          ELSE 0
        END
      ) AS has_ntb_purchase
    FROM
      TABLE(EXTEND_TIME_WINDOW('conversions_all_for_audiences', 'P30D', 'P0D'))
    WHERE
      event_subtype = 'order'
      AND user_id IS NOT NULL
    GROUP BY
      user_id
  ),

  labelled_users AS (
    SELECT
      eu.user_id,
      c.cutoff_date AS anchor_date,
      CASE
        WHEN n.has_ntb_purchase = 1
         AND SECONDS_BETWEEN(c.cutoff_date, n.first_ntb_purchase_date) BETWEEN 0 AND 86400 * {follow_up_days}
        THEN 1
        ELSE 0
      END AS label
    FROM eligible_users eu
    CROSS JOIN cutoff c
    LEFT JOIN ntb_users n ON eu.user_id = n.user_id
    WHERE NOT (COALESCE(n.has_ntb_purchase, 0) = 1 AND n.first_ntb_purchase_date <= c.cutoff_date)
  ),

  sampled_labelled_users AS (
    SELECT *
    FROM labelled_users
    WHERE label = 1 OR (label = 0 AND random() < {negative_keep_rate_sql})
  ),

  impression_features AS (
    SELECT
      lu.user_id,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (fbi.impression_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 THEN fbi.impressions
          ELSE 0
        END
      ) AS total_impressions_1d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (fbi.impression_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 3 THEN fbi.impressions
          ELSE 0
        END
      ) AS total_impressions_3d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (fbi.impression_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 5 THEN fbi.impressions
          ELSE 0
        END
      ) AS total_impressions_5d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (fbi.impression_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 7 THEN fbi.impressions
          ELSE 0
        END
      ) AS total_impressions_7d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (fbi.impression_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 14 THEN fbi.impressions
          ELSE 0
        END
      ) AS total_impressions_14d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (fbi.impression_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 30 THEN fbi.impressions
          ELSE 0
        END
      ) AS total_impressions_30d,
      COUNT(
        DISTINCT CASE
          WHEN SECONDS_BETWEEN(
            fbi.impression_dt_utc,
            lu.anchor_date
          ) BETWEEN 0 AND 86400
          THEN CAST(fbi.impression_dt_utc AS DATE)
          ELSE NULL
        END
      ) AS active_days_1d,
      COUNT(
        DISTINCT CASE
          WHEN SECONDS_BETWEEN(
            fbi.impression_dt_utc,
            lu.anchor_date
          ) BETWEEN 0 AND 86400 * 7
          THEN CAST(fbi.impression_dt_utc AS DATE)
          ELSE NULL
        END
      ) AS active_days_7d,
      COUNT(
        DISTINCT CASE
          WHEN SECONDS_BETWEEN(
            fbi.impression_dt_utc,
            lu.anchor_date
          ) BETWEEN 0 AND 86400 * 14
          THEN CAST(fbi.impression_dt_utc AS DATE)
          ELSE NULL
        END
      ) AS active_days_14d,
      MIN(
        CASE
          WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 30
          THEN SECONDS_BETWEEN(fbi.impression_dt_utc, lu.anchor_date)
          ELSE NULL
        END
      ) AS last_impression_seconds_ago_30d
    FROM
      sampled_labelled_users lu
      INNER JOIN filtered_brand_impressions fbi ON lu.user_id = fbi.user_id
    GROUP BY
      lu.user_id
  ),

  click_features AS (
    SELECT
      lu.user_id,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN(
            dc.click_dt_utc,
            lu.anchor_date
          ) BETWEEN 0 AND 86400
          THEN dc.clicks
          ELSE 0
        END
      ) AS total_clicks_1d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN(
            dc.click_dt_utc,
            lu.anchor_date
          ) BETWEEN 0 AND 86400 * 3
          THEN dc.clicks
          ELSE 0
        END
      ) AS total_clicks_3d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN(
            dc.click_dt_utc,
            lu.anchor_date
          ) BETWEEN 0 AND 86400 * 5
          THEN dc.clicks
          ELSE 0
        END
      ) AS total_clicks_5d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN(
            dc.click_dt_utc,
            lu.anchor_date
          ) BETWEEN 0 AND 86400 * 7
          THEN dc.clicks
          ELSE 0
        END
      ) AS total_clicks_7d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN(
            dc.click_dt_utc,
            lu.anchor_date
          ) BETWEEN 0 AND 86400 * 14
          THEN dc.clicks
          ELSE 0
        END
      ) AS total_clicks_14d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN(
            dc.click_dt_utc,
            lu.anchor_date
          ) BETWEEN 0 AND 86400 * 30
          THEN dc.clicks
          ELSE 0
        END
      ) AS total_clicks_30d
    FROM
      sampled_labelled_users lu
      LEFT JOIN filtered_brand_clicks dc ON lu.user_id = dc.user_id
    GROUP BY
      lu.user_id
  ),

  conversion_features AS (
    SELECT
      lu.user_id,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400
          AND ca.event_subtype = 'shoppingCart' THEN 1
          ELSE 0
        END
      ) AS total_add_to_cart_1d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 3
          AND ca.event_subtype = 'shoppingCart' THEN 1
          ELSE 0
        END
      ) AS total_add_to_cart_3d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 5
          AND ca.event_subtype = 'shoppingCart' THEN 1
          ELSE 0
        END
      ) AS total_add_to_cart_5d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 7
          AND ca.event_subtype = 'shoppingCart' THEN 1
          ELSE 0
        END
      ) AS total_add_to_cart_7d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 14
          AND ca.event_subtype = 'shoppingCart' THEN 1
          ELSE 0
        END
      ) AS total_add_to_cart_14d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 30
          AND ca.event_subtype = 'shoppingCart' THEN 1
          ELSE 0
        END
      ) AS total_add_to_cart_30d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400
          AND ca.event_subtype = 'detailPageView' THEN 1
          ELSE 0
        END
      ) AS total_dpv_1d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 3
          AND ca.event_subtype = 'detailPageView' THEN 1
          ELSE 0
        END
      ) AS total_dpv_3d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 5
          AND ca.event_subtype = 'detailPageView' THEN 1
          ELSE 0
        END
      ) AS total_dpv_5d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 7
          AND ca.event_subtype = 'detailPageView' THEN 1
          ELSE 0
        END
      ) AS total_dpv_7d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 14
          AND ca.event_subtype = 'detailPageView' THEN 1
          ELSE 0
        END
      ) AS total_dpv_14d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 30
          AND ca.event_subtype = 'detailPageView' THEN 1
          ELSE 0
        END
      ) AS total_dpv_30d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400
          AND ca.event_subtype = 'wishList' THEN 1
          ELSE 0
        END
      ) AS total_wishlist_1d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 3
          AND ca.event_subtype = 'wishList' THEN 1
          ELSE 0
        END
      ) AS total_wishlist_3d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 5
          AND ca.event_subtype = 'wishList' THEN 1
          ELSE 0
        END
      ) AS total_wishlist_5d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 7
          AND ca.event_subtype = 'wishList' THEN 1
          ELSE 0
        END
      ) AS total_wishlist_7d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 14
          AND ca.event_subtype = 'wishList' THEN 1
          ELSE 0
        END
      ) AS total_wishlist_14d,
      SUM(
        CASE
          WHEN SECONDS_BETWEEN (ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 30
          AND ca.event_subtype = 'wishList' THEN 1
          ELSE 0
        END
      ) AS total_wishlist_30d,
      MIN(
        CASE
          WHEN SECONDS_BETWEEN(ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 30
          THEN SECONDS_BETWEEN(ca.event_dt_utc, lu.anchor_date)
          ELSE NULL
        END
      ) AS last_conversion_seconds_ago_30d,
      MIN(
        CASE
          WHEN SECONDS_BETWEEN(ca.event_dt_utc, lu.anchor_date) BETWEEN 0 AND 86400 * 30
          AND ca.event_subtype = 'shoppingCart'
          THEN SECONDS_BETWEEN(ca.event_dt_utc, lu.anchor_date)
          ELSE NULL
        END
      ) AS last_cart_seconds_ago_30d
    FROM
      sampled_labelled_users lu
      LEFT JOIN TABLE(EXTEND_TIME_WINDOW('conversions_all_for_audiences', 'P30D', 'P0D')) ca ON lu.user_id = ca.user_id
    GROUP BY
      lu.user_id
  ),

  inmarket_user_segments AS (
    SELECT DISTINCT
      lu.user_id,
      im.segment_id,
      im.segment_marketplace_id
    FROM
      sampled_labelled_users lu
      INNER JOIN audience_segments_eu_inmarket_for_audiences im
        ON lu.user_id = im.user_id
        AND im.segment_marketplace_id = {int(segment_marketplace_id)}
    WHERE
      im.segment_id IS NOT NULL
  ),

  user_inmarket_segments AS (
    SELECT
      ius.user_id,
      COUNT(DISTINCT ius.segment_id) AS inmarket_segment_count,
      COUNT(DISTINCT sm.category_level_1) AS inmarket_category_l1_count,
      COUNT(DISTINCT sm.category_level_2) AS inmarket_category_l2_count
    FROM
      inmarket_user_segments ius
      LEFT JOIN segment_metadata_for_audiences sm
        ON ius.segment_id = sm.segment_id
        AND ius.segment_marketplace_id = sm.segment_marketplace_id
    GROUP BY
      ius.user_id
  )

SELECT
  lu.user_id,
  COALESCE(imp.total_impressions_1d, 0) AS total_impressions_1d,
  COALESCE(imp.total_impressions_3d, 0) AS total_impressions_3d,
  COALESCE(imp.total_impressions_5d, 0) AS total_impressions_5d,
  COALESCE(imp.total_impressions_7d, 0) AS total_impressions_7d,
  COALESCE(imp.total_impressions_14d, 0) AS total_impressions_14d,
  GREATEST(COALESCE(imp.total_impressions_3d, 0) - COALESCE(imp.total_impressions_1d, 0), 0) AS bucket_impressions_1_3d,
  GREATEST(COALESCE(imp.total_impressions_7d, 0) - COALESCE(imp.total_impressions_3d, 0), 0) AS bucket_impressions_3_7d,
  GREATEST(COALESCE(imp.total_impressions_14d, 0) - COALESCE(imp.total_impressions_7d, 0), 0) AS bucket_impressions_7_14d,
  COALESCE(clk.total_clicks_1d, 0) AS total_clicks_1d,
  COALESCE(clk.total_clicks_3d, 0) AS total_clicks_3d,
  COALESCE(clk.total_clicks_5d, 0) AS total_clicks_5d,
  COALESCE(clk.total_clicks_7d, 0) AS total_clicks_7d,
  COALESCE(clk.total_clicks_14d, 0) AS total_clicks_14d,
  COALESCE(clk.total_clicks_30d, 0) AS total_clicks_30d,
  GREATEST(COALESCE(clk.total_clicks_3d, 0) - COALESCE(clk.total_clicks_1d, 0), 0) AS bucket_clicks_1_3d,
  GREATEST(COALESCE(clk.total_clicks_7d, 0) - COALESCE(clk.total_clicks_3d, 0), 0) AS bucket_clicks_3_7d,
  GREATEST(COALESCE(clk.total_clicks_14d, 0) - COALESCE(clk.total_clicks_7d, 0), 0) AS bucket_clicks_7_14d,
  GREATEST(COALESCE(clk.total_clicks_30d, 0) - COALESCE(clk.total_clicks_14d, 0), 0) AS bucket_clicks_14_30d,
  COALESCE(conv.total_add_to_cart_1d, 0) AS total_add_to_cart_1d,
  COALESCE(conv.total_add_to_cart_3d, 0) AS total_add_to_cart_3d,
  COALESCE(conv.total_add_to_cart_5d, 0) AS total_add_to_cart_5d,
  COALESCE(conv.total_add_to_cart_7d, 0) AS total_add_to_cart_7d,
  COALESCE(conv.total_add_to_cart_14d, 0) AS total_add_to_cart_14d,
  COALESCE(conv.total_add_to_cart_30d, 0) AS total_add_to_cart_30d,
  GREATEST(COALESCE(conv.total_add_to_cart_3d, 0) - COALESCE(conv.total_add_to_cart_1d, 0), 0) AS bucket_add_to_cart_1_3d,
  GREATEST(COALESCE(conv.total_add_to_cart_7d, 0) - COALESCE(conv.total_add_to_cart_3d, 0), 0) AS bucket_add_to_cart_3_7d,
  GREATEST(COALESCE(conv.total_add_to_cart_14d, 0) - COALESCE(conv.total_add_to_cart_7d, 0), 0) AS bucket_add_to_cart_7_14d,
  GREATEST(COALESCE(conv.total_add_to_cart_30d, 0) - COALESCE(conv.total_add_to_cart_14d, 0), 0) AS bucket_add_to_cart_14_30d,
  COALESCE(conv.total_dpv_1d, 0) AS total_dpv_1d,
  COALESCE(conv.total_dpv_3d, 0) AS total_dpv_3d,
  COALESCE(conv.total_dpv_5d, 0) AS total_dpv_5d,
  COALESCE(conv.total_dpv_7d, 0) AS total_dpv_7d,
  COALESCE(conv.total_dpv_14d, 0) AS total_dpv_14d,
  COALESCE(conv.total_dpv_30d, 0) AS total_dpv_30d,
  COALESCE(conv.total_wishlist_1d, 0) AS total_wishlist_1d,
  COALESCE(conv.total_wishlist_3d, 0) AS total_wishlist_3d,
  COALESCE(conv.total_wishlist_5d, 0) AS total_wishlist_5d,
  COALESCE(conv.total_wishlist_7d, 0) AS total_wishlist_7d,
  COALESCE(conv.total_wishlist_14d, 0) AS total_wishlist_14d,
  COALESCE(conv.total_wishlist_30d, 0) AS total_wishlist_30d,
  COALESCE(imp.active_days_1d, 0) AS active_days_1d,
  COALESCE(imp.active_days_7d, 0) AS active_days_7d,
  COALESCE(imp.active_days_14d, 0) AS active_days_14d,
  COALESCE(conv.total_add_to_cart_30d, 0) / NULLIF(COALESCE(clk.total_clicks_30d, 0), 0) AS cart_to_click_rate_30d,
  COALESCE(conv.total_dpv_30d, 0) / NULLIF(COALESCE(imp.total_impressions_30d, 0), 0) AS dpv_to_impression_rate_30d,
  COALESCE(imp.last_impression_seconds_ago_30d, 86400 * 31) AS last_impression_seconds_ago_30d,
  COALESCE(conv.last_conversion_seconds_ago_30d, 86400 * 31) AS last_conversion_seconds_ago_30d,
  COALESCE(conv.last_cart_seconds_ago_30d, 86400 * 31) AS last_cart_seconds_ago_30d,
  COALESCE(ims.inmarket_segment_count, 0) AS inmarket_segment_count,
  COALESCE(ims.inmarket_category_l1_count, 0) AS inmarket_category_l1_count,
  COALESCE(ims.inmarket_category_l2_count, 0) AS inmarket_category_l2_count,
  lu.label
FROM
  sampled_labelled_users lu
  LEFT JOIN impression_features imp ON lu.user_id = imp.user_id
  LEFT JOIN click_features clk ON lu.user_id = clk.user_id
  LEFT JOIN conversion_features conv ON lu.user_id = conv.user_id
  LEFT JOIN user_inmarket_segments ims ON lu.user_id = ims.user_id
'''

train_sql = " ".join(train_sql_base.strip().rstrip(";").split())


print("Campaign filter inserted correctly:")
print("{campaign_filter_condition_di}" not in train_sql)
print("In-market segment marketplace filter:", segment_marketplace_id)

print("\nBeginning of generated SQL:")
print(train_sql[:1000])

model_tag = "ntb"  # change to "cltv" for CLTV runs
brand_tag = brand_config.brand_tag
market_tag = str(target_market).strip().lower()
type_tag = 'test_run' #change to final for the very final model training run

dry_run_payload_training = {
    "workflow_type": "train", ### Supported options are: (train, inference)
    "data_type": "MMM", ### This field is REQUIRED for measurement use cases. It is not used for audience use cases
    "training_parameters": {
        "name": acm_algorithm_name + '_training_model',
        "instance_type": "ml.m5.24xlarge",
        "description": f"{acm_algorithm_name} Training model",
        "vol_size": 60,
        "timeout": 259200,
        "tags": [
            {"Key": "model", "Value": model_tag},
            {"Key": "market", "Value": market_tag},
            {"Key": "type", "Value": type_tag},
            {"Key": "brand", "Value": brand_tag}
        ],
      "environment": {
            'SM_CHANNEL_TRAIN1': '/opt/ml/input/data/train/',
          'AG_TIME_LIMIT': '14400',
          'AG_PRESETS': 'best_quality',
          'AG_EVAL_METRIC': 'average_precision',
          'AG_SLIM_ARTIFACTS': 'true',
          'AG_MODEL_SIZE_CAP_MB': '2048',
          'AG_EVAL_SAMPLE_ROWS': '100000',
          'AG_TEST_FRACTION': '0.20',
          'AG_NEG_SAMPLE_RATIO': '4',
          'AG_MAX_TEST_ROWS': '5000000',
          'AG_USE_SAMPLE_WEIGHT': 'true',
          'AG_RANDOM_SEED': '42',
          'METRIC_LOG_REPEATS': '50',
          'HC_TOP_PERCENT': '1.0',
          'DECISION_THRESHOLD' : '0.20',
            'BRAND_MAPPING_SOURCE': 'glue',
            'BRAND_MAPPING_DATABASE': 'custom-audience-pilot-db',
            'BRAND_MAPPING_TABLE': 'brand_resolved_campaign_data',
            'BRAND_MAPPING_WORKGROUP': 'primary',
            'BRAND_MAPPING_REGION': 'us-east-1',
            'CAMPAIGN_COLUMN': 'campaign',
            'BRAND_COLUMN': 'brand',
            'BRAND_MAPPING_CAMPAIGN_COLUMN': 'campaign',
            'BRAND_MAPPING_BRAND_COLUMN': 'brand',
            'BRAND_FILTER_LIST': brand_config.brand_filter_list
        }
    },
    "training_input_channels": [
        {
            "name": "train",
            "sql_query": train_sql,
            "time_start": "2025-10-13T00:00:00",
            "time_end":"2026-09-13T00:00:00"
        }
    ] 
}

print("Training SQL prepared with SQL-level fallback filtering.")
print(train_sql[:1200] + "..." if len(train_sql) > 1200 else train_sql)

Campaign filter inserted correctly:
True
In-market segment marketplace filter: 7

Beginning of generated SQL:
WITH cutoff AS ( SELECT EPOCH_TO_UTC_TIMESTAMP( SECONDS_BETWEEN(CAST('1970-01-01' AS DATE), BUILT_IN_PARAMETER('TIME_WINDOW_END')) - (86400 * 28) ) AS cutoff_date ), filtered_brand_impressions AS ( SELECT di.user_id, di.impression_dt_utc, di.impressions FROM TABLE(EXTEND_TIME_WINDOW('dsp_impressions_for_audiences', 'P30D', 'P0D')) di WHERE di.user_id IS NOT NULL AND ((di.campaign IN ('Big Spring Event-Apdeo-Dove_Added Value', 'Big Spring Event-Hair-Dove_Added Value', 'Big Spring Event-Skin Cleansing-Dove_Added Value', 'CN005471-Ecomm-Digretail-Deo-(Dove)_DEO_Dove_Disp_Cons_Core-En-Mba-2026-Amz-Prg-P3C099V-O-D2Hiedc', 'CN005471-Ecomm-Digretail-Deo-(Dove)_DEO_Dove_Disp_Conv_Core-En-Mba-2026-Amz-Prg-P3C09Cf-O-D2Hiedc', 'CN005471_Dove_DEO_SBV_KW_Cons_Whole-Body-Deo-Combo-Brand-Phrase', 'CN005471_Dove_DEO_SBV_KW_Cons_Whole-Body-Deo-Combo-Brand-Phrase-CPC', 'CN005471_Dove_DEO_SBV_KW_

## Step 2.2: Start Training Pipeline ⚠️ INCURS AWS COSTS

This step initiates the complete training workflow:

### What Happens When You Run This:
1. **Container Build** (~1 min): Builds Docker image from your `ml/code/train/` folder
2. **ECR Push** (~1 min): Uploads container to AWS Elastic Container Registry
3. **ML Input Channel Creation**: Executes your SQL queries against AMC data
4. **SageMaker Training**: Runs your training algorithm on the prepared data
5. **Model Storage**: Saves trainedModelId for use in inference

### Monitoring & Costs:
- **Monitor Progress**: Use the AWS Console Step Functions URL provided after execution
- **Default Cost**: <~$1 for dry-run with sample data
- **Cost Factors**: Instance type, data volume, training duration
- **Error Handling**: Step Functions will show detailed error messages if issues occur

In [10]:
# Start the pipeline with the payload
import os

# Use Finch explicitly for local container build/push
os.environ['ACM_CONTAINER_CLI'] = 'finch'
# Force x86_64-compatible container images for SageMaker ml.m5 instances
os.environ['ACM_CONTAINER_PLATFORM'] = 'linux/amd64'

# Ensure container CLI binaries are discoverable from notebook kernel process
for bin_dir in ['/usr/local/bin', '/Users/ecrawfor/.docker/bin']:
    if bin_dir not in os.environ.get('PATH', ''):
        os.environ['PATH'] = bin_dir + os.pathsep + os.environ.get('PATH', '')

print('container cli:', os.environ.get('ACM_CONTAINER_CLI'))
print('container platform:', os.environ.get('ACM_CONTAINER_PLATFORM'))
print('finch in PATH:', os.popen('command -v finch').read().strip() or 'not found')

train_result = pipeline.start_pipeline(
    dry_run_payload_training,
    schema_file_path=schema_file_path,
    cmaa_config_directory=cmaa_config_directory,
    metrics_definition_directory=metrics_definition_directory,
 )

print('Started training execution ARN:', train_result.get('execution_arn'))

container cli: finch
container platform: linux/amd64
finch in PATH: /usr/local/bin/finch
Starting ACM pipeline for algorithm: ntb_audience_ca_dove_v2
Workflow type: train

Checking to see if default images exist
Image 980468996155.dkr.ecr.us-east-1.amazonaws.com/amc-acm-custom-model-stack-ca-acm-repo:ntb_audience_ca_dove_v2_train_image already exists, skipping build
Image 980468996155.dkr.ecr.us-east-1.amazonaws.com/amc-acm-custom-model-stack-ca-acm-repo:ntb_audience_ca_dove_v2_inference_image already exists, skipping build

Step 1 & 2: Building and pushing container...
Building Docker container...
Container name: ntb_audience_ca_dove_v2_train_container
Container CLI: finch
Container platform: linux/amd64
train Docker container build completed successfully
Container successfully pushed to: 980468996155.dkr.ecr.us-east-1.amazonaws.com/amc-acm-custom-model-stack-ca-acm-repo:ntb_audience_ca_dove_v2_train_image

Step 3: Creating model associations...
Skipping creating model association as 

In [65]:
# Diagnostics: inspect ECR repo config fallback candidates
print('cloud_formation_env_vars ecr_repo_name:', cloud_formation_env_vars.get('ecr_repo_name'))
import boto3
ecr = boto3.client('ecr', region_name='us-east-1')
repos = [r['repositoryName'] for r in ecr.describe_repositories().get('repositories', [])]
print('ECR repos count:', len(repos))
for r in repos:
    print('-', r)

cloud_formation_env_vars ecr_repo_name: None
ECR repos count: 4
- amc-acm-custom-model-stack-acm-repo
- amc-acm-custom-model-stack-ca-acm-repo
- acm-ntb-train
- acm-ntb-infer


In [72]:
# Diagnostics: confirm Step Functions ARN resolution
cloud_formation_env_vars = get_cloudformation_env_vars()
print('resolved step_function_arn:', cloud_formation_env_vars.get('step_function_arn'))
print('pipeline.step_function_arn:', getattr(pipeline, 'step_function_arn', None))

resolved step_function_arn: arn:aws:states:us-east-1:980468996155:stateMachine:amc-acm-custom-model-stack-ca-ACM-Pipeline
pipeline.step_function_arn: arn:aws:states:us-east-1:980468996155:stateMachine:amc-acm-custom-model-stack-ca-ACM-Pipeline


In [74]:
# Diagnostics: who is this notebook authenticated as?
import boto3, json
sts = boto3.client('sts')
identity = sts.get_caller_identity()
print(json.dumps(identity, indent=2))
arn = identity.get('Arn', '')
print('is_sagemaker_assumed_role:', ':assumed-role/' in arn and 'sagemaker' in arn.lower())

{
  "UserId": "AIDA6ISEBTA5T6CSWWOZF",
  "Account": "980468996155",
  "Arn": "arn:aws:iam::980468996155:user/emma.crawford@unilever.com",
  "ResponseMetadata": {
    "RequestId": "0f571b2e-ec09-442b-be8b-0919607d80f4",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "0f571b2e-ec09-442b-be8b-0919607d80f4",
      "x-amz-sts-extended-request-id": "MTp1cy1lYXN0LTE6UzoxNzkyNjc1MzE1Mzc5OlI6MnRIREpaYXE=",
      "content-type": "text/xml",
      "content-length": "423",
      "date": "Thu, 22 Oct 2026 13:21:55 GMT"
    },
    "RetryAttempts": 0
  }
}
is_sagemaker_assumed_role: False


In [86]:
# Diagnostics: inspect recent execution record sizes in DynamoDB
import json
records = pipeline.get_algorithm_executions(limit=5)
print('recent execution count:', len(records))

table = pipeline.dynamodb.Table(pipeline.dynamodb_table_executions)
for rec in records:
    ts = rec.get('execution_timestamp')
    item = table.get_item(Key={'acm_algorithm_name': pipeline.acm_algorithm_name, 'execution_timestamp': ts}).get('Item', {})
    payload_len = len(item.get('payload') or '')
    train_channels_len = len(item.get('training_channels') or '')
    infer_channels_len = len(item.get('inference_channels') or '')
    schema_len = len(item.get('schema_used') or '')
    approx_json_bytes = len(json.dumps(item, default=str).encode('utf-8'))
    print('---')
    print('execution_timestamp:', ts)
    print('status:', item.get('status'))
    print('workflow_type:', item.get('workflow_type'))
    print('payload_len:', payload_len)
    print('training_channels_len:', train_channels_len)
    print('inference_channels_len:', infer_channels_len)
    print('schema_len:', schema_len)
    print('approx_item_bytes:', approx_json_bytes)

recent execution count: 5
---
execution_timestamp: 2026-10-22T15:48:20.131416
status: FAILED
workflow_type: train
payload_len: 560
training_channels_len: 146
inference_channels_len: 0
schema_len: 41
approx_item_bytes: 2262
---
execution_timestamp: 2026-10-22T14:18:11.052148
status: FAILED
workflow_type: train
payload_len: 176548
training_channels_len: 174614
inference_channels_len: 0
schema_len: 231
approx_item_bytes: 353114
---
execution_timestamp: 2026-10-16T09:01:09.046744
status: COMPLETED
workflow_type: inference
payload_len: 176506
training_channels_len: 0
inference_channels_len: 175154
schema_len: 231
approx_item_bytes: 353923
---
execution_timestamp: 2026-10-15T15:51:48.267288
status: COMPLETED
workflow_type: train
payload_len: 176548
training_channels_len: 174614
inference_channels_len: 0
schema_len: 231
approx_item_bytes: 353117
---
execution_timestamp: 2026-10-15T13:31:35.587229
status: COMPLETED
workflow_type: train
payload_len: 176493
training_channels_len: 174614
inferenc

In [82]:
# Diagnostics: largest string fields in latest execution record
import json
latest = records[0]
latest_ts = latest.get('execution_timestamp')
item = table.get_item(Key={'acm_algorithm_name': pipeline.acm_algorithm_name, 'execution_timestamp': latest_ts}).get('Item', {})
sizes = []
for k, v in item.items():
    if isinstance(v, str):
        sizes.append((k, len(v)))
sizes.sort(key=lambda x: x[1], reverse=True)
print('latest execution:', latest_ts)
print('top string field sizes:')
for k, n in sizes[:12]:
    print(f'{k}: {n}')
print('approx_item_bytes:', len(json.dumps(item, default=str).encode('utf-8')))

latest execution: 2026-10-22T15:48:20.131416
top string field sizes:
payload: 560
console_url: 202
training_channels: 146
execution_arn: 122
schema_used: 41
training_description: 38
training_model_name: 38
last_updated: 26
execution_timestamp: 26
acm_algorithm_name: 23
training_instance_type: 14
status: 7
approx_item_bytes: 1961


In [85]:
pipeline.display_error_summary()

✅ No recent errors found for algorithm: ntb_audience_ca_dove_v2


In [ ]:
#Stop pipeline
# pipeline.stop_pipeline(train_result['execution_arn'])

## Step 2.3: Retrieve Training Results

After training completes successfully, extract the trained model ID and metadata:

### What This Returns:
- **trained_model_id**: Unique identifier for your trained model (needed for inference)
- **create_time**: Timestamp for retrieving training metrics
- **status**: Confirmation that training completed successfully

**Note**: This cell will only work after the Step Function shows 'SUCCEEDED' status.

## Step 2.4: Evaluate Model Performance

Review your model's training metrics to decide if it's ready for inference:



These metrics are automatically logged during training and stored in CloudWatch.

In [14]:
train_pipeline_results = pipeline.get_pipeline_status(train_result['execution_arn'])
print(json.dumps(train_pipeline_results, indent=2))
try:
    trainedModelId=train_pipeline_results['trained_model_id']
    start_time = train_pipeline_results['create_time']
    print("Trained Model ID and start_time updated")
except:
    print('trainedModelId is not set')

{
  "status": null,
  "instance_id": null,
  "use_case_type": null,
  "number_of_processed_channels": 0,
  "workflow_type": null
}
trainedModelId is not set


In [13]:
# Diagnostics: inspect final Step Functions failure event details
import boto3
import json

execution_arn = train_result['execution_arn']
sfn = boto3.client('stepfunctions', region_name='us-east-1')

history = sfn.get_execution_history(
    executionArn=execution_arn,
    reverseOrder=True,
    maxResults=50
)

events = history.get('events', [])

# Print top-level failed event details
failed_events = [
    e for e in events
    if e.get('type') in ('ExecutionFailed', 'TaskFailed', 'LambdaFunctionFailed')
]

print('execution_arn:', execution_arn)
print('failed_event_count:', len(failed_events))

for idx, e in enumerate(failed_events[:8], 1):
    print('\n--- failed event', idx, '---')
    print('id:', e.get('id'))
    print('type:', e.get('type'))
    print('previousEventId:', e.get('previousEventId'))
    if 'executionFailedEventDetails' in e:
        print('executionFailedEventDetails:', json.dumps(e['executionFailedEventDetails'], indent=2))
    if 'taskFailedEventDetails' in e:
        d = e['taskFailedEventDetails']
        print('taskFailedEventDetails.error:', d.get('error'))
        cause = d.get('cause', '')
        print('taskFailedEventDetails.cause_preview:', cause[:2000])
    if 'lambdaFunctionFailedEventDetails' in e:
        d = e['lambdaFunctionFailedEventDetails']
        print('lambdaFunctionFailedEventDetails.error:', d.get('error'))
        cause = d.get('cause', '')
        print('lambdaFunctionFailedEventDetails.cause_preview:', cause[:2000])


execution_arn: arn:aws:states:us-east-1:980468996155:execution:amc-acm-custom-model-stack-ca-ACM-Pipeline:ntb_audience_ca_dove_v210251252
failed_event_count: 0


In [88]:
# Diagnostics: show event chain around final failure
import boto3
import json

execution_arn = train_result['execution_arn']
sfn = boto3.client('stepfunctions', region_name='us-east-1')

all_events = []
next_token = None
while True:
    kwargs = {'executionArn': execution_arn, 'maxResults': 1000}
    if next_token:
        kwargs['nextToken'] = next_token
    page = sfn.get_execution_history(**kwargs)
    all_events.extend(page.get('events', []))
    next_token = page.get('nextToken')
    if not next_token:
        break

by_id = {e['id']: e for e in all_events}
final = max(all_events, key=lambda e: e['id'])
print('final_event_id:', final['id'], 'type:', final['type'])

# Walk back previousEventId chain for context
cursor = final
for _ in range(12):
    print('\n=== event ===')
    print('id:', cursor.get('id'), 'type:', cursor.get('type'), 'prev:', cursor.get('previousEventId'))
    # print interesting detail keys only
    for k in [
        'stateEnteredEventDetails',
        'stateExitedEventDetails',
        'taskScheduledEventDetails',
        'taskStartedEventDetails',
        'taskFailedEventDetails',
        'taskSucceededEventDetails',
        'lambdaFunctionFailedEventDetails',
        'executionFailedEventDetails'
    ]:
        if k in cursor:
            val = cursor[k]
            txt = json.dumps(val, indent=2, default=str)
            print(k + ':')
            print(txt[:3000])
    prev = cursor.get('previousEventId')
    if not prev or prev not in by_id:
        break
    cursor = by_id[prev]


final_event_id: 124 type: ExecutionFailed

=== event ===
id: 124 type: ExecutionFailed prev: 123
executionFailedEventDetails:
{
  "error": "ACMjobError",
  "cause": "ACM Training job failed - check logs for more details"
}

=== event ===
id: 123 type: FailStateEntered prev: 122
stateEnteredEventDetails:
{
  "name": "JobFailedExit",
  "input": "{\"ExecutedVersion\":\"$LATEST\",\"Payload\":{\"status\":\"ok\",\"executionId\":null,\"manifestUri\":null,\"trainedModelId\":\"0122353648d-bf50-4539-96c5-7d33e2c3bd3b\",\"workflow_type\":\"train\",\"instance_id\":\"amcy3dkn05n\",\"use_case_type\":\"audience\",\"number_of_processed_channels\":1},\"SdkHttpMetadata\":{\"AllHttpHeaders\":{\"X-Amz-Executed-Version\":[\"$LATEST\"],\"x-amzn-Remapped-Content-Length\":[\"0\"],\"Connection\":[\"keep-alive\"],\"x-amzn-RequestId\":[\"2388a33d-0210-4544-8ce0-65797ac91134\"],\"Content-Length\":[\"239\"],\"Date\":[\"Thu, 22 Oct 2026 15:37:55 GMT\"],\"X-Amzn-Trace-Id\":[\"Root=1-6ada2dd2-2dbc1aa13c5de7ca67571286

In [89]:
# Diagnostics: locate first state that introduced FAILED status or ACMjobError markers
import boto3
import json

execution_arn = train_result['execution_arn']
sfn = boto3.client('stepfunctions', region_name='us-east-1')

all_events = []
next_token = None
while True:
    kwargs = {'executionArn': execution_arn, 'maxResults': 1000}
    if next_token:
        kwargs['nextToken'] = next_token
    page = sfn.get_execution_history(**kwargs)
    all_events.extend(page.get('events', []))
    next_token = page.get('nextToken')
    if not next_token:
        break

markers = []
for e in all_events:
    blob = json.dumps(e, default=str)
    if '"FAILED"' in blob or 'ACMjobError' in blob or 'error' in blob.lower():
        markers.append((e.get('id'), e.get('type'), e.get('previousEventId')))

print('marker_count:', len(markers))
for m in markers[:20]:
    print('event_id:', m[0], 'type:', m[1], 'prev:', m[2])

# Show details for the earliest interesting marker
if markers:
    target_id = markers[0][0]
    target = next(ev for ev in all_events if ev.get('id') == target_id)
    print('\nEarliest marker event details (trimmed):')
    text = json.dumps(target, indent=2, default=str)
    print(text[:6000])


marker_count: 13
event_id: 107 type: TaskSucceeded prev: 106
event_id: 108 type: TaskStateExited prev: 107
event_id: 109 type: ChoiceStateEntered prev: 108
event_id: 110 type: ChoiceStateExited prev: 109
event_id: 111 type: PassStateEntered prev: 110
event_id: 112 type: PassStateExited prev: 111
event_id: 113 type: TaskStateEntered prev: 112
event_id: 114 type: TaskScheduled prev: 113
event_id: 116 type: TaskSucceeded prev: 115
event_id: 117 type: TaskStateExited prev: 116
event_id: 118 type: TaskStateEntered prev: 117
event_id: 119 type: TaskScheduled prev: 118
event_id: 124 type: ExecutionFailed prev: 123

Earliest marker event details (trimmed):
{
  "timestamp": "2026-10-22 16:37:50.577000+01:00",
  "type": "TaskSucceeded",
  "id": 107,
  "previousEventId": 106,
  "taskSucceededEventDetails": {
    "resourceType": "lambda",
    "resource": "invoke",
    "output": "{\"ExecutedVersion\":\"$LATEST\",\"Payload\":{\"statusCode\":400,\"body\":{\"message\":\"Training failed. Error message 

In [90]:
# Diagnostics: inspect ECR image manifest platform for the current train image
import boto3
import json

repo = pipeline.ecr_repo_name
tag = f"{pipeline.acm_algorithm_name}_train_image"
region = pipeline.region

ecr = boto3.client("ecr", region_name=region)
resp = ecr.batch_get_image(
    repositoryName=repo,
    imageIds=[{"imageTag": tag}],
    acceptedMediaTypes=[
        "application/vnd.docker.distribution.manifest.list.v2+json",
        "application/vnd.oci.image.index.v1+json",
        "application/vnd.docker.distribution.manifest.v2+json",
        "application/vnd.oci.image.manifest.v1+json",
    ],
)

if resp.get("failures"):
    print("ECR failures:", resp["failures"])
else:
    img = resp["images"][0]
    media = img.get("imageManifestMediaType")
    manifest = json.loads(img["imageManifest"])
    print("repository:", repo)
    print("tag:", tag)
    print("manifest media type:", media)
    if "manifests" in manifest:
        print("platform entries:")
        for m in manifest["manifests"]:
            p = m.get("platform", {})
            print("-", p.get("os"), p.get("architecture"), p.get("variant"))
    else:
        print("single manifest (no multi-platform index); architecture not explicit here")


repository: amc-acm-custom-model-stack-ca-acm-repo
tag: ntb_audience_ca_dove_v2_train_image
manifest media type: application/vnd.docker.distribution.manifest.v2+json
single manifest (no multi-platform index); architecture not explicit here


In [11]:
# Diagnostics: resolve architecture from image config blob
import json
import urllib.request

repo = pipeline.ecr_repo_name
tag = f"{pipeline.acm_algorithm_name}_train_image"
region = pipeline.region

ecr = boto3.client("ecr", region_name=region)
resp = ecr.batch_get_image(
    repositoryName=repo,
    imageIds=[{"imageTag": tag}],
    acceptedMediaTypes=["application/vnd.docker.distribution.manifest.v2+json"]
)

img = resp["images"][0]
manifest = json.loads(img["imageManifest"])
config_digest = manifest["config"]["digest"]

url_resp = ecr.get_download_url_for_layer(repositoryName=repo, layerDigest=config_digest)
url = url_resp["downloadUrl"]
with urllib.request.urlopen(url) as r:
    cfg = json.loads(r.read().decode("utf-8"))

print("repo:", repo)
print("tag:", tag)
print("config digest:", config_digest)
print("os:", cfg.get("os"))
print("architecture:", cfg.get("architecture"))
print("variant:", cfg.get("variant"))


repo: amc-acm-custom-model-stack-ca-acm-repo
tag: ntb_audience_ca_dove_v2_train_image
config digest: sha256:67a0dc39b6dd006a90b36ced7fee20d56ee651fcdd4477bbad804fee5252fd16
os: linux
architecture: amd64
variant: None


In [23]:
# Diagnostics: resolve architecture from image config blob (inference image)
import json
import urllib.request

repo = pipeline.ecr_repo_name
tag = f"{pipeline.acm_algorithm_name}_inference_image"
region = pipeline.region

ecr = boto3.client("ecr", region_name=region)
resp = ecr.batch_get_image(
    repositoryName=repo,
    imageIds=[{"imageTag": tag}],
    acceptedMediaTypes=["application/vnd.docker.distribution.manifest.v2+json"]
)

img = resp["images"][0]
manifest = json.loads(img["imageManifest"])
config_digest = manifest["config"]["digest"]

url_resp = ecr.get_download_url_for_layer(repositoryName=repo, layerDigest=config_digest)
url = url_resp["downloadUrl"]
with urllib.request.urlopen(url) as r:
    cfg = json.loads(r.read().decode("utf-8"))

print("repo:", repo)
print("tag:", tag)
print("config digest:", config_digest)
print("os:", cfg.get("os"))
print("architecture:", cfg.get("architecture"))
print("variant:", cfg.get("variant"))

repo: amc-acm-custom-model-stack-ca-acm-repo
tag: ntb_audience_ca_dove_v2_inference_image
config digest: sha256:c82b0f40bf7c17807d2272f390f91d91627f99ff1804eb325cfdb7cd8f7b6370
os: linux
architecture: amd64
variant: None


In [38]:
pipeline.get_trained_model_metrics_agg(trainedModelId, start_time)

{'Train Set Positive Rate': None,
 'Recall': None,
 'Test Positive Rate': None,
 'HC Precision (10_0)': None,
 'Test Split Positive Rate': None,
 'Columns': None,
 'Accuracy': None,
 'Average Precision Lift': None,
 'Predicted Positive Rate': None,
 'ROC AUC': None,
 'Average Precision': None,
 'HC Top Percent (10_0)': None,
 'MCC': None,
 'Precision': None,
 'Predicted Positive Count': None,
 'HC Lift (10_0)': None}

## Step 2.5: Decision Point

Based on your model metrics, decide your next action:

### If Metrics Look Good:
Proceed to the Inference section below to generate predictions on new data.

### If Metrics Need Improvement:
Consider these options:
- **Modify SQL queries**: Include more/different features or adjust time windows
- **Update training code**: Modify your algorithm in `ml/code/train/train.py`
- **Adjust parameters**: Change instance types, hyperparameters, or training duration
- **Re-run training**: Use the same algorithm name to update your model

**Tip**: Each training run is tracked in DynamoDB

# Part 3: Model Inference

Generate predictions using your trained model on new AMC data.

## Execution History
All previous training and inference runs are automatically tracked in DynamoDB, including:
- Trained model IDs
- Execution timestamps
- Configuration parameters
- Success/failure status

## Step 3.1: Configure Inference Parameters

Inference Jobs leverage a Trained Model to generate predictions (inference) on input data (provided via ML Input Channel). The output destination depends on your use case type:

- **Audience Use Cases**: Results can be uploaded to a new AMC dataset and leveraged for audience creation
- **Measurement Use Cases**: Results are saved directly to S3 bucket for analysis

[Inferencing](https://docs.aws.amazon.com/clean-rooms/latest/userguide/run-inference-jobs.html) in AWS Clean Rooms ML leverages the [SageMaker Batch Transform](https://docs.aws.amazon.com/sagemaker/latest/dg/batch-transform.html) API. The AMC Custom Models QuickStart guide includes instructions for setting up a basic inferencing container. As described in the SageMaker guide for [creating your own inference container](https://docs.aws.amazon.com/sagemaker/latest/dg/adapt-inference-container.html), the container must host a webserver which responds to incoming requests. The incoming request contains batches of data from the ML Input Channel in CSV format. The inferencing container should load the trained model parameters from **opt/ml/model** (saved during training), and respond to inferencing requests by adding additional columns generated by the model.

#### Usage
- Inference jobs accept an ML Input Channel, Trained Model, and CMA (inference container) as input. The purpose of an inference job is to augment the input data (ML Input Channel) with additional columns containing the model-generated predictions. 
- **For Audience Use Cases**: The inference job should produce CSV files containing a user_id column, and additional columns which represent the model-generated predictions. The dataset generated during inference can then be uploaded to a new AMC dataset for audience creation.
- **For Measurement Use Cases**: The inference job output is saved directly to S3 for measurement analysis and MMM workflows.

#### Tips

- Each Trained Model Inference Job accepts single ML Input Channel (SageMaker limitation). In order generate inference results on large datasets, it is recommended to split the input data into batches (multiple ML Input Channels) and create multiple Trained Model Inference Jobs. The results from each inference job can then be loaded into AMC for audience creation. - See the AMC Custom Models QuickStart guides for a sample inferencing container.


### Key Configuration:
- **trained_model_id**: The model ID from your training step
- **SQL Query**: Define what new data to score (different from training data)
- **Time Window**: Typically more recent data than your training period
- **Compute Resources**: Usually similar to training, but can be adjusted
- **instance_count**: Unlike training where it must be 1, for inference you can set this up to 10 to enable parallel processing. It's recommended to set this to match the numberOfFiles field from your ML Input Channel to optimize parallel processing.
- **vol_size**: The size of the AWS EBS (Elastic Block Store) volume in GB. This volume stores both the input data from ML Input Channels and any model artifacts written during training. You should size this based on the approximateSizeOnDisk of your ML Input Channels plus additional storage needed for model artifacts.

Optional:
- **environment**: Key-value pairs that will be made available as environment variables to your training container. In the example, 'SM_CHANNEL_TRAIN': 'mlic_name' maps the ML Input Channel name to a specific training channel.


**Important**: Inference only supports 1 input channel (unlike training which supports up to 20).

In [20]:
# Inference payload aligned to training feature logic (inference version).
if "campaign_filter_condition_campaign" not in globals():
    raise ValueError(
        "Run Part 1.3 first to build SQL campaign filter conditions."
    )

infer_sql_base = r'''
WITH
  cutoff AS (
    SELECT BUILT_IN_PARAMETER('TIME_WINDOW_END') AS cutoff_date
  ),

  filtered_brand_impressions AS (
    SELECT
      di.user_id,
      di.impression_dt_utc,
      di.impressions
    FROM
      TABLE(EXTEND_TIME_WINDOW('dsp_impressions_for_audiences', 'P30D', 'P0D')) di
    WHERE
      di.user_id IS NOT NULL
      AND {campaign_filter_condition_di}
  ),

  filtered_brand_clicks AS (
    SELECT
      dc.user_id,
      dc.click_dt_utc,
      dc.clicks
    FROM
      TABLE(EXTEND_TIME_WINDOW('dsp_clicks_for_audiences', 'P30D', 'P0D')) dc
    WHERE
      dc.user_id IS NOT NULL
      AND {campaign_filter_condition_campaign}
  ),

  ntb_users AS (
    SELECT
      user_id,
      MIN(CASE WHEN new_to_brand = TRUE THEN event_dt_utc END) AS first_ntb_purchase_date,
      MAX(CASE WHEN new_to_brand = TRUE THEN 1 ELSE 0 END) AS has_ntb_purchase
    FROM
      TABLE(EXTEND_TIME_WINDOW('conversions_all_for_audiences', 'P30D', 'P0D'))
    WHERE
      event_subtype = 'order'
      AND user_id IS NOT NULL
    GROUP BY
      user_id
  ),

  eligible_users AS (
    SELECT DISTINCT
      fbi.user_id
    FROM
      filtered_brand_impressions fbi
    CROSS JOIN cutoff c
    LEFT JOIN ntb_users n ON fbi.user_id = n.user_id
    WHERE
      SECONDS_BETWEEN(fbi.impression_dt_utc, c.cutoff_date) BETWEEN 0 AND 86400 * 30
      AND NOT (COALESCE(n.has_ntb_purchase, 0) = 1 AND n.first_ntb_purchase_date <= c.cutoff_date)
  ),

  inference_users AS (
    SELECT
      eu.user_id,
      c.cutoff_date AS anchor_date
    FROM
      eligible_users eu
    CROSS JOIN cutoff c
  ),

  impression_features AS (
    SELECT
      iu.user_id,
      SUM(CASE WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 THEN fbi.impressions ELSE 0 END) AS total_impressions_1d,
      SUM(CASE WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 3 THEN fbi.impressions ELSE 0 END) AS total_impressions_3d,
      SUM(CASE WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 5 THEN fbi.impressions ELSE 0 END) AS total_impressions_5d,
      SUM(CASE WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 7 THEN fbi.impressions ELSE 0 END) AS total_impressions_7d,
      SUM(CASE WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 14 THEN fbi.impressions ELSE 0 END) AS total_impressions_14d,
      SUM(CASE WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 30 THEN fbi.impressions ELSE 0 END) AS total_impressions_30d,
      COUNT(DISTINCT CASE WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 THEN CAST(fbi.impression_dt_utc AS DATE) ELSE NULL END) AS active_days_1d,
      COUNT(DISTINCT CASE WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 7 THEN CAST(fbi.impression_dt_utc AS DATE) ELSE NULL END) AS active_days_7d,
      COUNT(DISTINCT CASE WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 14 THEN CAST(fbi.impression_dt_utc AS DATE) ELSE NULL END) AS active_days_14d,
      MIN(CASE WHEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 30 THEN SECONDS_BETWEEN(fbi.impression_dt_utc, iu.anchor_date) ELSE NULL END) AS last_impression_seconds_ago_30d
    FROM
      inference_users iu
      INNER JOIN filtered_brand_impressions fbi ON iu.user_id = fbi.user_id
    GROUP BY
      iu.user_id
  ),

  click_features AS (
    SELECT
      iu.user_id,
      SUM(CASE WHEN SECONDS_BETWEEN(dc.click_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 THEN dc.clicks ELSE 0 END) AS total_clicks_1d,
      SUM(CASE WHEN SECONDS_BETWEEN(dc.click_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 3 THEN dc.clicks ELSE 0 END) AS total_clicks_3d,
      SUM(CASE WHEN SECONDS_BETWEEN(dc.click_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 5 THEN dc.clicks ELSE 0 END) AS total_clicks_5d,
      SUM(CASE WHEN SECONDS_BETWEEN(dc.click_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 7 THEN dc.clicks ELSE 0 END) AS total_clicks_7d,
      SUM(CASE WHEN SECONDS_BETWEEN(dc.click_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 14 THEN dc.clicks ELSE 0 END) AS total_clicks_14d,
      SUM(CASE WHEN SECONDS_BETWEEN(dc.click_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 30 THEN dc.clicks ELSE 0 END) AS total_clicks_30d
    FROM
      inference_users iu
      LEFT JOIN filtered_brand_clicks dc ON iu.user_id = dc.user_id
    GROUP BY
      iu.user_id
  ),

  conversion_features AS (
    SELECT
      iu.user_id,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 AND ca.event_subtype = 'shoppingCart' THEN 1 ELSE 0 END) AS total_add_to_cart_1d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 3 AND ca.event_subtype = 'shoppingCart' THEN 1 ELSE 0 END) AS total_add_to_cart_3d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 5 AND ca.event_subtype = 'shoppingCart' THEN 1 ELSE 0 END) AS total_add_to_cart_5d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 7 AND ca.event_subtype = 'shoppingCart' THEN 1 ELSE 0 END) AS total_add_to_cart_7d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 14 AND ca.event_subtype = 'shoppingCart' THEN 1 ELSE 0 END) AS total_add_to_cart_14d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 30 AND ca.event_subtype = 'shoppingCart' THEN 1 ELSE 0 END) AS total_add_to_cart_30d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 AND ca.event_subtype = 'detailPageView' THEN 1 ELSE 0 END) AS total_dpv_1d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 3 AND ca.event_subtype = 'detailPageView' THEN 1 ELSE 0 END) AS total_dpv_3d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 5 AND ca.event_subtype = 'detailPageView' THEN 1 ELSE 0 END) AS total_dpv_5d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 7 AND ca.event_subtype = 'detailPageView' THEN 1 ELSE 0 END) AS total_dpv_7d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 14 AND ca.event_subtype = 'detailPageView' THEN 1 ELSE 0 END) AS total_dpv_14d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 30 AND ca.event_subtype = 'detailPageView' THEN 1 ELSE 0 END) AS total_dpv_30d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 AND ca.event_subtype = 'wishList' THEN 1 ELSE 0 END) AS total_wishlist_1d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 3 AND ca.event_subtype = 'wishList' THEN 1 ELSE 0 END) AS total_wishlist_3d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 5 AND ca.event_subtype = 'wishList' THEN 1 ELSE 0 END) AS total_wishlist_5d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 7 AND ca.event_subtype = 'wishList' THEN 1 ELSE 0 END) AS total_wishlist_7d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 14 AND ca.event_subtype = 'wishList' THEN 1 ELSE 0 END) AS total_wishlist_14d,
      SUM(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 30 AND ca.event_subtype = 'wishList' THEN 1 ELSE 0 END) AS total_wishlist_30d,
      MIN(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 30 THEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) ELSE NULL END) AS last_conversion_seconds_ago_30d,
      MIN(CASE WHEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) BETWEEN 0 AND 86400 * 30 AND ca.event_subtype = 'shoppingCart' THEN SECONDS_BETWEEN(ca.event_dt_utc, iu.anchor_date) ELSE NULL END) AS last_cart_seconds_ago_30d
    FROM
      inference_users iu
      LEFT JOIN TABLE(EXTEND_TIME_WINDOW('conversions_all_for_audiences', 'P30D', 'P0D')) ca ON iu.user_id = ca.user_id
    GROUP BY
      iu.user_id
  ),

  inmarket_user_segments AS (
    SELECT DISTINCT
      iu.user_id,
      im.segment_id,
      im.segment_marketplace_id
    FROM
      inference_users iu
      INNER JOIN audience_segments_eu_inmarket_for_audiences im
        ON iu.user_id = im.user_id
        AND im.segment_marketplace_id = {int(segment_marketplace_id)}
    WHERE
      im.segment_id IS NOT NULL
  ),

  user_inmarket_segments AS (
    SELECT
      ius.user_id,
      COUNT(DISTINCT ius.segment_id) AS inmarket_segment_count,
      COUNT(DISTINCT sm.category_level_1) AS inmarket_category_l1_count,
      COUNT(DISTINCT sm.category_level_2) AS inmarket_category_l2_count
    FROM
      inmarket_user_segments ius
      LEFT JOIN segment_metadata_for_audiences sm
        ON ius.segment_id = sm.segment_id
        AND ius.segment_marketplace_id = sm.segment_marketplace_id
    GROUP BY
      ius.user_id
  )

SELECT
  iu.user_id,
  COALESCE(imp.total_impressions_1d, 0) AS total_impressions_1d,
  COALESCE(imp.total_impressions_3d, 0) AS total_impressions_3d,
  COALESCE(imp.total_impressions_5d, 0) AS total_impressions_5d,
  COALESCE(imp.total_impressions_7d, 0) AS total_impressions_7d,
  COALESCE(imp.total_impressions_14d, 0) AS total_impressions_14d,
  COALESCE(imp.total_impressions_30d, 0) AS total_impressions_30d,
  GREATEST(COALESCE(imp.total_impressions_3d, 0) - COALESCE(imp.total_impressions_1d, 0), 0) AS bucket_impressions_1_3d,
  GREATEST(COALESCE(imp.total_impressions_7d, 0) - COALESCE(imp.total_impressions_3d, 0), 0) AS bucket_impressions_3_7d,
  GREATEST(COALESCE(imp.total_impressions_14d, 0) - COALESCE(imp.total_impressions_7d, 0), 0) AS bucket_impressions_7_14d,
  COALESCE(clk.total_clicks_1d, 0) AS total_clicks_1d,
  COALESCE(clk.total_clicks_3d, 0) AS total_clicks_3d,
  COALESCE(clk.total_clicks_5d, 0) AS total_clicks_5d,
  COALESCE(clk.total_clicks_7d, 0) AS total_clicks_7d,
  COALESCE(clk.total_clicks_14d, 0) AS total_clicks_14d,
  COALESCE(clk.total_clicks_30d, 0) AS total_clicks_30d,
  GREATEST(COALESCE(clk.total_clicks_3d, 0) - COALESCE(clk.total_clicks_1d, 0), 0) AS bucket_clicks_1_3d,
  GREATEST(COALESCE(clk.total_clicks_7d, 0) - COALESCE(clk.total_clicks_3d, 0), 0) AS bucket_clicks_3_7d,
  GREATEST(COALESCE(clk.total_clicks_14d, 0) - COALESCE(clk.total_clicks_7d, 0), 0) AS bucket_clicks_7_14d,
  GREATEST(COALESCE(clk.total_clicks_30d, 0) - COALESCE(clk.total_clicks_14d, 0), 0) AS bucket_clicks_14_30d,
  COALESCE(conv.total_add_to_cart_1d, 0) AS total_add_to_cart_1d,
  COALESCE(conv.total_add_to_cart_3d, 0) AS total_add_to_cart_3d,
  COALESCE(conv.total_add_to_cart_5d, 0) AS total_add_to_cart_5d,
  COALESCE(conv.total_add_to_cart_7d, 0) AS total_add_to_cart_7d,
  COALESCE(conv.total_add_to_cart_14d, 0) AS total_add_to_cart_14d,
  COALESCE(conv.total_add_to_cart_30d, 0) AS total_add_to_cart_30d,
  GREATEST(COALESCE(conv.total_add_to_cart_3d, 0) - COALESCE(conv.total_add_to_cart_1d, 0), 0) AS bucket_add_to_cart_1_3d,
  GREATEST(COALESCE(conv.total_add_to_cart_7d, 0) - COALESCE(conv.total_add_to_cart_3d, 0), 0) AS bucket_add_to_cart_3_7d,
  GREATEST(COALESCE(conv.total_add_to_cart_14d, 0) - COALESCE(conv.total_add_to_cart_7d, 0), 0) AS bucket_add_to_cart_7_14d,
  GREATEST(COALESCE(conv.total_add_to_cart_30d, 0) - COALESCE(conv.total_add_to_cart_14d, 0), 0) AS bucket_add_to_cart_14_30d,
  COALESCE(conv.total_dpv_1d, 0) AS total_dpv_1d,
  COALESCE(conv.total_dpv_3d, 0) AS total_dpv_3d,
  COALESCE(conv.total_dpv_5d, 0) AS total_dpv_5d,
  COALESCE(conv.total_dpv_7d, 0) AS total_dpv_7d,
  COALESCE(conv.total_dpv_14d, 0) AS total_dpv_14d,
  COALESCE(conv.total_dpv_30d, 0) AS total_dpv_30d,
  COALESCE(conv.total_wishlist_1d, 0) AS total_wishlist_1d,
  COALESCE(conv.total_wishlist_3d, 0) AS total_wishlist_3d,
  COALESCE(conv.total_wishlist_5d, 0) AS total_wishlist_5d,
  COALESCE(conv.total_wishlist_7d, 0) AS total_wishlist_7d,
  COALESCE(conv.total_wishlist_14d, 0) AS total_wishlist_14d,
  COALESCE(conv.total_wishlist_30d, 0) AS total_wishlist_30d,
  COALESCE(imp.active_days_1d, 0) AS active_days_1d,
  COALESCE(imp.active_days_7d, 0) AS active_days_7d,
  COALESCE(imp.active_days_14d, 0) AS active_days_14d,
  COALESCE(conv.total_add_to_cart_30d, 0) / NULLIF(COALESCE(clk.total_clicks_30d, 0), 0) AS cart_to_click_rate_30d,
  COALESCE(conv.total_dpv_30d, 0) / NULLIF(COALESCE(imp.total_impressions_30d, 0), 0) AS dpv_to_impression_rate_30d,
  COALESCE(imp.last_impression_seconds_ago_30d, 86400 * 31) AS last_impression_seconds_ago_30d,
  COALESCE(conv.last_conversion_seconds_ago_30d, 86400 * 31) AS last_conversion_seconds_ago_30d,
  COALESCE(conv.last_cart_seconds_ago_30d, 86400 * 31) AS last_cart_seconds_ago_30d,
  COALESCE(ims.inmarket_segment_count, 0) AS inmarket_segment_count,
  COALESCE(ims.inmarket_category_l1_count, 0) AS inmarket_category_l1_count,
  COALESCE(ims.inmarket_category_l2_count, 0) AS inmarket_category_l2_count
FROM
  inference_users iu
  LEFT JOIN impression_features imp ON iu.user_id = imp.user_id
  LEFT JOIN click_features clk ON iu.user_id = clk.user_id
  LEFT JOIN conversion_features conv ON iu.user_id = conv.user_id
  LEFT JOIN user_inmarket_segments ims ON iu.user_id = ims.user_id
'''

# SQL-level filtering only, aligned with training behavior (mapped allowlist + case-insensitive LIKE fallback).
sql = infer_sql_base.strip().rstrip(";")

sql = sql.replace("{campaign_filter_condition_di}", (campaign_filter_condition_di or "TRUE").strip())
sql = sql.replace("{campaign_filter_condition_campaign}", (campaign_filter_condition_campaign or "TRUE").strip())
sql = sql.replace("{int(segment_marketplace_id)}", str(int(segment_marketplace_id)))

if "{" in sql or "}" in sql:
    raise ValueError("Unresolved SQL placeholders remain")

infer_sql = sql

NTB_payload_inference = {
    "workflow_type": "inference",
    "inference_parameters": {
        "name": acm_algorithm_name + "_inference_model",
        "trained_model_id": trainedModelId,
        "instance_type": "ml.m5.24xlarge",
        "instance_count": 1,
        "vol_size": 10,
        "environment": {
            "AG_TOP_PERCENT": "10",
            'BRAND_MAPPING_SOURCE': 'glue',
            'BRAND_MAPPING_DATABASE': 'custom-audience-pilot-db',
            'BRAND_MAPPING_TABLE': 'brand_resolved_campaign_data',
            'BRAND_MAPPING_WORKGROUP': 'primary',
            'BRAND_MAPPING_REGION': 'us-east-1',
            'CAMPAIGN_COLUMN': 'campaign',
            'BRAND_COLUMN': 'brand',
            'BRAND_MAPPING_CAMPAIGN_COLUMN': 'campaign',
            'BRAND_MAPPING_BRAND_COLUMN': 'brand',
            'BRAND_FILTER_LIST': brand_config.brand_filter_list
        },
    },
    "inference_input_channels": [
        {
            "name": "NTB_input_1",
            "sql_query": infer_sql,
            "time_start": "2026-09-15T00:00:00Z",
            "time_end": "2026-10-15T00:00:00Z"
        }
    ]
}

print("Inference SQL prepared with training-aligned feature logic and SQL-level brand filtering.")
print(infer_sql[:1200] + "..." if len(infer_sql) > 1200 else infer_sql)

Inference SQL prepared with training-aligned feature logic and SQL-level brand filtering.
WITH
  cutoff AS (
    SELECT BUILT_IN_PARAMETER('TIME_WINDOW_END') AS cutoff_date
  ),

  filtered_brand_impressions AS (
    SELECT
      di.user_id,
      di.impression_dt_utc,
      di.impressions
    FROM
      TABLE(EXTEND_TIME_WINDOW('dsp_impressions_for_audiences', 'P30D', 'P0D')) di
    WHERE
      di.user_id IS NOT NULL
      AND ((di.campaign IN ('Big Spring Event-Apdeo-Dove_Added Value', 'Big Spring Event-Hair-Dove_Added Value', 'Big Spring Event-Skin Cleansing-Dove_Added Value', 'CN005471-Ecomm-Digretail-Deo-(Dove)_DEO_Dove_Disp_Cons_Core-En-Mba-2026-Amz-Prg-P3C099V-O-D2Hiedc', 'CN005471-Ecomm-Digretail-Deo-(Dove)_DEO_Dove_Disp_Conv_Core-En-Mba-2026-Amz-Prg-P3C09Cf-O-D2Hiedc', 'CN005471_Dove_DEO_SBV_KW_Cons_Whole-Body-Deo-Combo-Brand-Phrase', 'CN005471_Dove_DEO_SBV_KW_Cons_Whole-Body-Deo-Combo-Brand-Phrase-CPC', 'CN005471_Dove_DEO_SBV_KW_Cons_Whole-Body-Deo-Combo-Generic-Exact', 'CN00

## Step 3.2: Start Inference Pipeline ⚠️ INCURS AWS COSTS

This step runs your trained model on new data and integrates results with AMC:

### What Happens When You Run This:
1. **Container Build** (~1 min): Builds Docker image from your `ml/code/infer/` folder
2. **ECR Push** (~1 min): Uploads container to AWS Elastic Container Registry
3. **Data Preparation**: Creates ML Input Channel with your inference SQL
4. **Model Inference**: Runs predictions using your trained model
5. **Output Processing**:
   - **Audience Use Cases**: Creates new dataset in AMC with predictions (~5-10 min), uploads to AMC tables (~5-15 min)
   - **Measurement Use Cases**: Saves results directly to S3 bucket for analysis

### What You Get:
- **Prediction Results**: Model scores for each user/entity in your query
- **Output Destination**:
  - **Audience**: New dataset available in AMC UI under 'Advertiser Uploaded', formatted for audience creation in Amazon DSP
  - **Measurement**: Results saved to S3 bucket for measurement analysis and MMM workflows

### Monitoring & Costs:
- **Monitor Progress**: Use AWS Console URL provided
- **Default Cost**: ~$1 for dry-run
- **Total Duration**: ~30-60 minutes end-to-end

In [22]:
# Start inference with the same container settings as training
import os

# Keep local build/push behavior consistent with successful training run
os.environ['ACM_CONTAINER_CLI'] = 'finch'
os.environ['ACM_CONTAINER_PLATFORM'] = 'linux/amd64'

for bin_dir in ['/usr/local/bin', '/Users/ecrawfor/.docker/bin']:
    if bin_dir not in os.environ.get('PATH', ''):
        os.environ['PATH'] = bin_dir + os.pathsep + os.environ.get('PATH', '')

print('container cli:', os.environ.get('ACM_CONTAINER_CLI'))
print('container platform:', os.environ.get('ACM_CONTAINER_PLATFORM'))
print('finch in PATH:', os.popen('command -v finch').read().strip() or 'not found')

infer_result = pipeline.start_pipeline(
    NTB_payload_inference,
    schema_file_path=schema_file_path,
    cmaa_config_directory=cmaa_config_directory,
    metrics_definition_directory=metrics_definition_directory,
 )
print('Started inference execution ARN:', infer_result.get('execution_arn'))

container cli: finch
container platform: linux/amd64
finch in PATH: /usr/local/bin/finch
Schema in file path
Starting ACM pipeline for algorithm: ntb_audience_ca_dove_v2
Workflow type: inference

Checking to see if default images exist
Image 980468996155.dkr.ecr.us-east-1.amazonaws.com/amc-acm-custom-model-stack-ca-acm-repo:ntb_audience_ca_dove_v2_train_image already exists, skipping build
Image 980468996155.dkr.ecr.us-east-1.amazonaws.com/amc-acm-custom-model-stack-ca-acm-repo:ntb_audience_ca_dove_v2_inference_image already exists, skipping build

Step 1 & 2: Building and pushing container...
Building Docker container...
Container name: ntb_audience_ca_dove_v2_inference_container
Container CLI: finch
Container platform: linux/amd64
inference Docker container build completed successfully
Container successfully pushed to: 980468996155.dkr.ecr.us-east-1.amazonaws.com/amc-acm-custom-model-stack-ca-acm-repo:ntb_audience_ca_dove_v2_inference_image

Step 3: Creating model associations...
Ski

In [92]:
import importlib
import src.ACMPipelineClass as acm_mod
importlib.reload(acm_mod)
from src.ACMPipelineClass import *
print('Reloaded ACMPipelineClass from disk')

Reloaded ACMPipelineClass from disk


## Step 3.3: Retrieve Inference Results

After inference completes, get the job details and output status:

### What This Returns:
- **trainedModelInferenceJobId**: Unique ID for this inference run
- **create_time**: Timestamp for tracking
- **status**: Confirmation of successful completion

**Note**: This cell will only work after the Step Function shows 'SUCCEEDED' status.

In [ ]:
infer_pipeline_results = pipeline.get_pipeline_status(infer_result['execution_arn'])
print(json.dumps(infer_pipeline_results, indent=2))

# Part 4: Access Your Results

Your model predictions are now available! Here's how to find and use them based on your use case:

## For Audience Use Cases:
### In AMC UI:
1. **Navigate to Datasets**: Look under 'Advertiser Uploaded' datasets
2. **Find Your Dataset**: Named `modeled_dataset_{your-algorithm-name}`
3. **Dataset Type**: This is an audiences-only dataset (not queryable for measurement)

### Next Steps:
- **Create Audiences**: Use the 'Manual ACM Notebooks/AMC Audiences.ipynb' notebook
- **Audience Types**: Create rule-based or look-alike audiences
- **Activation**: Audiences will be available in Amazon DSP for campaign targeting

## For Measurement Use Cases:
### In S3 Bucket:
1. **Access Results**: Model output is saved directly to your designated S3 bucket
2. **File Format**: Results are in CSV format with model predictions
3. **Analysis Ready**: Data is formatted for measurement analysis and MMM workflows

### Next Steps:
- **Download Results**: Access the S3 bucket to retrieve your model output files
- **Analysis**: Use the results for measurement analysis, attribution modeling, or MMM workflows
- **Integration**: Integrate results with your existing measurement and analytics tools

# APPENDIX - Troubleshooting & Advanced Usage

## Common Issues:
#### Training/Inference Fails:
- Check Step Functions console for detailed error messages
- Verify SQL queries work in AMC Query Editor first
- Ensure time windows have sufficient data
- **For Audience Use Cases**: Verify queries use '_for_audiences' datasources
- **For Measurement Use Cases**: Regular AMC datasources are supported

#### Use Case Configuration:
- **Wrong Output Destination**: Verify `use_case_type` is set correctly ('audience' vs 'measurement')
- **Path Issues**: Ensure `base_path` points to correct code directory
- **Metrics Issues**: Verify `metrics_name` matches your use case type

#### Cost Management:
- Choose appropriate instance types
- Monitor AWS billing dashboard
- **Measurement Use Cases**: May have different cost patterns due to S3 storage

#### Other: 
- AWS Console links provided for monitoring
- DynamoDB tables track all executions and configurations
- Both use case types support the same monitoring and debugging tools

In [16]:
# Recover latest successful training execution and trainedModelId
records = pipeline.get_algorithm_executions(limit=20)
train_records = [r for r in records if r.get('workflow_type') == 'train']
print('train_records:', len(train_records))

latest_success_arn = None
for r in train_records:
    arn = r.get('execution_arn')
    if not arn:
        continue
    st = pipeline.get_pipeline_status(arn)
    if st.get('status') == 'SUCCEEDED' and st.get('trained_model_id'):
        latest_success_arn = arn
        trainedModelId = st['trained_model_id']
        start_time = st.get('create_time')
        print('using execution:', arn)
        print('trainedModelId:', trainedModelId)
        break

if not latest_success_arn:
    raise RuntimeError('No successful training execution with trained_model_id found yet.')

train_records: 5


RuntimeError: No successful training execution with trained_model_id found yet.

In [17]:
# Inspect recent train executions and their parsed statuses
for i, r in enumerate(train_records[:10], 1):
    arn = r.get('execution_arn')
    print('\n--- train record', i, '---')
    print('execution_timestamp:', r.get('execution_timestamp'))
    print('execution_arn:', arn)
    st = pipeline.get_pipeline_status(arn) if arn else {}
    print('status:', st.get('status'))
    print('trained_model_id:', st.get('trained_model_id'))
    print('create_time:', st.get('create_time'))


--- train record 1 ---
execution_timestamp: 2026-10-25T12:52:54.761837
execution_arn: arn:aws:states:us-east-1:980468996155:execution:amc-acm-custom-model-stack-ca-ACM-Pipeline:ntb_audience_ca_dove_v210251252
status: None
trained_model_id: None
create_time: None

--- train record 2 ---
execution_timestamp: 2026-10-22T15:48:20.131416
execution_arn: arn:aws:states:us-east-1:980468996155:execution:amc-acm-custom-model-stack-ca-ACM-Pipeline:ntb_audience_ca_dove_v210221548
status: FAILED
trained_model_id: None
create_time: None

--- train record 3 ---
execution_timestamp: 2026-10-22T14:18:11.052148
execution_arn: arn:aws:states:us-east-1:980468996155:execution:amc-acm-custom-model-stack-ca-ACM-Pipeline:ntb_audience_ca_dove_v210221418
status: FAILED
trained_model_id: None
create_time: None

--- train record 4 ---
execution_timestamp: 2026-10-15T15:51:48.267288
execution_arn: arn:aws:states:us-east-1:980468996155:execution:amc-acm-custom-model-stack-ca-ACM-Pipeline:ntb_audience_ca_dove_v2101

In [18]:
# Direct Step Functions status/output check for latest training execution
import boto3, json
sfn = boto3.client('stepfunctions', region_name='us-east-1')
latest_train_arn = train_records[0]['execution_arn']
desc = sfn.describe_execution(executionArn=latest_train_arn)
print('executionArn:', latest_train_arn)
print('status:', desc.get('status'))
print('startDate:', desc.get('startDate'))
print('stopDate:', desc.get('stopDate'))
raw_out = desc.get('output')
print('has_output:', bool(raw_out))
if raw_out:
    print('output_preview:', raw_out[:2000])

executionArn: arn:aws:states:us-east-1:980468996155:execution:amc-acm-custom-model-stack-ca-ACM-Pipeline:ntb_audience_ca_dove_v210251252
status: SUCCEEDED
startDate: 2026-10-25 12:52:55.692000+00:00
stopDate: 2026-10-25 14:40:18.095000+00:00
has_output: True
output_preview: {"ExecutedVersion":"$LATEST","Payload":{"status":"ok","executionId":null,"manifestUri":null,"trainedModelId":"012545671f2-a03b-4058-921f-de4382329e9b","workflow_type":"train","instance_id":"amcy3dkn05n","use_case_type":"audience","number_of_processed_channels":1},"SdkHttpMetadata":{"AllHttpHeaders":{"X-Amz-Executed-Version":["$LATEST"],"x-amzn-Remapped-Content-Length":["0"],"Connection":["keep-alive"],"x-amzn-RequestId":["09c1e974-1f06-481a-bdc1-a6dde8630324"],"Content-Length":["239"],"Date":["Sun, 25 Oct 2026 14:40:18 GMT"],"X-Amzn-Trace-Id":["Root=1-6ade14d0-0c6d37740df4969c1d0e20d9;Parent=57123274cd4d07be;Sampled=0;Lineage=1:95a18e09:0"],"Content-Type":["application/json"]},"HttpHeaders":{"Connection":"keep-alive

In [19]:
# Extract trainedModelId from Step Functions output
out = json.loads(desc.get('output') or '{}')
payload = out.get('Payload', {})
trainedModelId = payload.get('trainedModelId')
start_time = str(desc.get('startDate'))
print('trainedModelId:', trainedModelId)
print('start_time:', start_time)
if not trainedModelId:
    raise RuntimeError('trainedModelId not found in execution output.')

trainedModelId: 012545671f2-a03b-4058-921f-de4382329e9b
start_time: 2026-10-25 12:52:55.692000+00:00
